# 01. Exploración inicial de datos epidemiológicos y geográficos

## Objetivo

Explorar la estructura y calidad de las fuentes de datos
que se utilizarán para analizar la distribución espacial
del dengue en Argentina.

Se trabajará con dos fuentes:

- Datos epidemiológicos del Ministerio de Salud.
- Geometrías provinciales del servicio Georef.

En esta primera etapa se analizarán dimensiones, variables,
tipos de datos, valores faltantes y consistencia de ambas fuentes.

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

print("Pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)

Pandas: 3.0.6
GeoPandas: 1.2.0


### A. Carga de datos

In [2]:
# Directorio de trabajo actual: notebooks/
BASE_DIR = Path.cwd()

# Directorio principal del proyecto
PROJECT_DIR = BASE_DIR.parent

# Directorio donde se encuentran los datos originales
RAW_DIR = PROJECT_DIR / "data" / "raw"

# Rutas de los archivos
DENGUE_FILE = RAW_DIR / "informacion-publica-dengue-zika-nacional-se-1-a-52-de-2023-2024-06-10.csv"
MAP_FILE = RAW_DIR / "provincias.geojson"

# Verificamos que los archivos existan
assert DENGUE_FILE.exists(), f"No se encontró el archivo: {DENGUE_FILE}"
assert MAP_FILE.exists(), f"No se encontró el archivo: {MAP_FILE}"

print("Rutas verificadas correctamente.")

Rutas verificadas correctamente.


In [3]:
# Cargamos los datos epidemiológicos
df_dengue = pd.read_csv(
    DENGUE_FILE,
    sep=";",
    encoding="utf-8-sig"
)

# Cargamos los datos geográficos
gdf_provincias = gpd.read_file(MAP_FILE)

print("Archivos cargados correctamente.")

C:\Users\luisi\anaconda3\envs\openfda_ds\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Archivos cargados correctamente.


In [4]:
print("Dengue:", df_dengue.shape)
print("Provincias:", gdf_provincias.shape)

print("\nGeometría:")
print(gdf_provincias.geometry.geom_type.value_counts())

print("\nSistema de coordenadas:")
print(gdf_provincias.crs)

Dengue: (18228, 10)
Provincias: (24, 9)

Geometría:
Polygon         17
MultiPolygon     7
Name: count, dtype: int64

Sistema de coordenadas:
EPSG:4326


### B. Exploración inicial y control de calidad de datos

In [5]:
# Información general del conjunto de datos
df_dengue.info()

<class 'pandas.DataFrame'>
RangeIndex: 18228 entries, 0 to 18227
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   id_depto_indec_residencia  18228 non-null  int64
 1   departamento_residencia    18228 non-null  str  
 2   id_prov_indec_residencia   18228 non-null  int64
 3   provincia_residencia       18228 non-null  str  
 4   anio_min                   18228 non-null  int64
 5   evento                     18228 non-null  str  
 6   id_grupo_etario            18228 non-null  int64
 7   grupo_etario               18228 non-null  str  
 8   sepi_min                   18228 non-null  int64
 9   cantidad                   18228 non-null  int64
dtypes: int64(6), str(4)
memory usage: 2.1 MB


In [6]:
# Primeros registros
display(df_dengue.head())

# Nombres de las columnas
print(df_dengue.columns.tolist())

,id_depto_indec_residencia,departamento_residencia,id_prov_indec_residencia,provincia_residencia,anio_min,evento,id_grupo_etario,grupo_etario,sepi_min,cantidad
0,5,COMUNA 5,2,CABA,2023,Dengue,6,De 10 a 14 años,1,1
1,140,San Fernando,22,Chaco,2023,Dengue,8,De 20 a 24 años,1,1
2,28,Capital,54,Misiones,2023,Dengue,11,De 45 a 65 años,1,1
3,140,San Fernando,22,Chaco,2023,Dengue,5,De 5 a 9 años,1,2
4,9,COMUNA 9,2,CABA,2023,Dengue,11,De 45 a 65 años,1,1


['id_depto_indec_residencia', 'departamento_residencia', 'id_prov_indec_residencia', 'provincia_residencia', 'anio_min', 'evento', 'id_grupo_etario', 'grupo_etario', 'sepi_min', 'cantidad']


In [7]:
# Valores registrados en las variables categóricas y temporales
columnas_revision = [
    "anio_min",
    "evento",
    "sepi_min",
    "provincia_residencia"
]

for columna in columnas_revision:
    print(f"\n{columna}:")
    print(sorted(df_dengue[columna].unique()))


anio_min:
[np.int64(2023)]

evento:
['Dengue']

sepi_min:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52)]

provincia_residencia:
['Buenos Aires', 'CABA', 'Catamarca', 'Chaco', 'Chubut', 'Corrientes', 'Córdoba', 'Entre Ríos', 'Formosa', 'Jujuy', 'La Pampa', 'La Rioja', 'Mendoza', 'Misiones', 'Neuquén', 'Río Negro', 'Salta', 'S

In [8]:
# Estadísticas descriptivas de la variable cantidad
df_dengue["cantidad"].describe()

count    18228.000000
mean         8.057713
std         22.386103
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max        495.000000
Name: cantidad, dtype: float64

In [9]:
# Duplicados exactos
print("Duplicados exactos:", df_dengue.duplicated().sum())

# Duplicados según la combinación de variables que identifica cada registro
columnas_clave = [
    "id_depto_indec_residencia",
    "id_prov_indec_residencia",
    "anio_min",
    "evento",
    "id_grupo_etario",
    "sepi_min"
]

print(
    "Duplicados según combinación de claves:",
    df_dengue.duplicated(subset=columnas_clave).sum()
)

Duplicados exactos: 10
Duplicados según combinación de claves: 32


In [10]:
# Identificamos las filas con claves repetidas
duplicados_clave = df_dengue[
    df_dengue.duplicated(
        subset=columnas_clave,
        keep=False
    )
].sort_values(columnas_clave)

display(duplicados_clave)

,id_depto_indec_residencia,departamento_residencia,id_prov_indec_residencia,provincia_residencia,anio_min,evento,id_grupo_etario,grupo_etario,sepi_min,cantidad
4098,84,Guasayán,86,Santiago del Estero,2023,Dengue,12,Mayores de 65 años,13,1
4256,84,Capital,86,Santiago del Estero,2023,Dengue,12,Mayores de 65 años,13,1
3491,999,desconocido,14,Córdoba,2023,Dengue,9,De 25 a 34 años,13,1
4107,999,desconocido,14,Córdoba,2023,Dengue,9,De 25 a 34 años,13,1
6950,999,desconocido,14,Córdoba,2023,Dengue,9,De 25 a 34 años,16,1
...,...,...,...,...,...,...,...,...,...,...
7763,999,desconocido,99,desconocida,2023,Dengue,11,De 45 a 65 años,16,6
3418,999,desconocido,99,desconocida,2023,Dengue,12,Mayores de 65 años,13,2
3467,999,desconocido,99,desconocida,2023,Dengue,12,Mayores de 65 años,13,2
17048,999,desconocido,99,desconocida,2023,Dengue,12,Mayores de 65 años,49,1


In [11]:
# Separamos los duplicados exactos
duplicados_exactos = df_dengue[
    df_dengue.duplicated(keep=False)
].sort_values(columnas_clave)

display(duplicados_exactos)

,id_depto_indec_residencia,departamento_residencia,id_prov_indec_residencia,provincia_residencia,anio_min,evento,id_grupo_etario,grupo_etario,sepi_min,cantidad
3491,999,desconocido,14,Córdoba,2023,Dengue,9,De 25 a 34 años,13,1
4107,999,desconocido,14,Córdoba,2023,Dengue,9,De 25 a 34 años,13,1
18044,999,desconocido,34,Formosa,2023,Dengue,7,De 15 a 19 años,52,1
18123,999,desconocido,34,Formosa,2023,Dengue,7,De 15 a 19 años,52,1
2361,999,desconocido,90,Tucumán,2023,Dengue,4,De 2 a 4 años,12,2
2675,999,desconocido,90,Tucumán,2023,Dengue,4,De 2 a 4 años,12,2
6929,999,desconocido,90,Tucumán,2023,Dengue,4,De 2 a 4 años,16,2
7169,999,desconocido,90,Tucumán,2023,Dengue,4,De 2 a 4 años,16,2
8065,999,desconocido,90,Tucumán,2023,Dengue,4,De 2 a 4 años,17,2
8076,999,desconocido,90,Tucumán,2023,Dengue,4,De 2 a 4 años,17,2


In [12]:
# Casos registrados por provincia
casos_por_provincia = (
    df_dengue.groupby("provincia_residencia")["cantidad"]
    .sum()
    .sort_values(ascending=False)
)

display(casos_por_provincia)

provincia_residencia
Tucumán                24172
Santa Fe               21328
Chaco                  18369
Santiago del Estero    15843
Salta                  15145
CABA                   13292
Buenos Aires           10443
Córdoba                 8078
Formosa                 6898
Jujuy                   5410
Corrientes              3090
Misiones                1846
Catamarca               1073
Entre Ríos               696
desconocida              468
La Rioja                 401
Santa Cruz                71
San Luis                  65
Tierra del Fuego          49
Mendoza                   43
Neuquén                   39
La Pampa                  28
Río Negro                 12
Chubut                     9
San Juan                   8
Name: cantidad, dtype: int64

In [13]:
# Agrupamos por las claves que identifican cada registro
revision_duplicados = (
    df_dengue
    .groupby(columnas_clave)
    .agg(
        cantidad_filas=("cantidad", "size"),
        valores_cantidad=("cantidad", "nunique"),
        suma_cantidad=("cantidad", "sum")
    )
    .reset_index()
)

# Nos quedamos con las combinaciones repetidas
revision_duplicados = revision_duplicados[
    revision_duplicados["cantidad_filas"] > 1
]

# Identificamos las claves repetidas con cantidades diferentes
duplicados_diferentes = revision_duplicados[
    revision_duplicados["valores_cantidad"] > 1
]

display(duplicados_diferentes)

print("Combinaciones repetidas:", len(revision_duplicados))
print("Combinaciones con cantidades diferentes:", len(duplicados_diferentes))

,id_depto_indec_residencia,id_prov_indec_residencia,anio_min,evento,id_grupo_etario,sepi_min,cantidad_filas,valores_cantidad,suma_cantidad
17939,999,14,2023,Dengue,9,16,2,2,3
17979,999,90,2023,Dengue,4,8,2,2,3
17981,999,90,2023,Dengue,4,11,2,2,6
17983,999,90,2023,Dengue,4,13,2,2,6
17985,999,90,2023,Dengue,4,15,2,2,6
17998,999,90,2023,Dengue,5,15,2,2,10
18008,999,90,2023,Dengue,6,12,2,2,3
18019,999,90,2023,Dengue,11,12,2,2,3
18053,999,99,2023,Dengue,5,8,2,2,3
18071,999,99,2023,Dengue,6,14,2,2,3


Combinaciones repetidas: 32
Combinaciones con cantidades diferentes: 21


In [14]:
# Inspeccionamos todas las columnas de los registros repetidos
caso_revision = df_dengue[
    (df_dengue["id_depto_indec_residencia"] == 999) &
    (df_dengue["id_prov_indec_residencia"] == 14) &
    (df_dengue["anio_min"] == 2023) &
    (df_dengue["id_grupo_etario"] == 9) &
    (df_dengue["sepi_min"] == 16)
]

display(caso_revision)

,id_depto_indec_residencia,departamento_residencia,id_prov_indec_residencia,provincia_residencia,anio_min,evento,id_grupo_etario,grupo_etario,sepi_min,cantidad
6950,999,desconocido,14,Córdoba,2023,Dengue,9,De 25 a 34 años,16,1
7795,999,desconocido,14,Córdoba,2023,Dengue,9,De 25 a 34 años,16,2


### C. Criterios de depuración y tratamiento de registros

Se conservaron los registros originales, incluidos aquellos con combinaciones de claves repetidas. Dado que la documentación disponible no permite determinar si corresponden a duplicaciones o aportes independientes, no se eliminaron filas automáticamente. Para el análisis provincial, las cantidades se agregaron mediante suma. Los registros sin provincia identificada se excluyeron de la representación geográfica. Esta decisión se considera una limitación del análisis.

In [15]:
# Creamos una copia de trabajo sin modificar el archivo original
df_trabajo = df_dengue.copy()

print("Registros originales:", len(df_dengue))
print("Registros de trabajo:", len(df_trabajo))

Registros originales: 18228
Registros de trabajo: 18228


In [16]:
# Identificamos los registros sin provincia conocida
df_desconocida = df_trabajo[
    df_trabajo["id_prov_indec_residencia"] == 99
].copy()

# Conservamos los registros con provincia identificada
df_nacional = df_trabajo[
    df_trabajo["id_prov_indec_residencia"] != 99
].copy()

print("Casos de provincia desconocida:", df_desconocida["cantidad"].sum())
print("Casos con provincia identificada:", df_nacional["cantidad"].sum())

Casos de provincia desconocida: 468
Casos con provincia identificada: 146408


In [17]:
# Agregamos los casos por código y nombre de provincia
casos_provincia = (
    df_nacional
    .groupby(
        ["id_prov_indec_residencia", "provincia_residencia"],
        as_index=False
    )["cantidad"]
    .sum()
    .rename(columns={"cantidad": "casos"})
    .sort_values("casos", ascending=False)
)

display(casos_provincia)

,id_prov_indec_residencia,provincia_residencia,casos
22,90,Tucumán,24172
20,82,Santa Fe,21328
5,22,Chaco,18369
21,86,Santiago del Estero,15843
16,66,Salta,15145
0,2,CABA,13292
1,6,Buenos Aires,10443
3,14,Córdoba,8078
8,34,Formosa,6898
9,38,Jujuy,5410


### D. Datos poblacionales 

In [18]:
POBLACION_FILE = RAW_DIR / "proyecciones_jurisdicciones_2022_2040_c1.xlsx"
excel = pd.ExcelFile(POBLACION_FILE, engine="calamine")

print("Cantidad de hojas:", len(excel.sheet_names))
print("\nNombres de las primeras hojas:")
print(excel.sheet_names[:10])

Cantidad de hojas: 27

Nombres de las primeras hojas:
['GraphData', 'ÍNDICE DE CUADROS', '01-TOTAL DEL PAÍS', '02-CABA', '06-BUENOS AIRES', '10-CATAMARCA', '14-CÓRDOBA', '18-CORRIENTES', '22-CHACO', '26-CHUBUT']


In [19]:
# Excluir hojas que no corresponden a jurisdicciones
hojas_jurisdicciones = [
    hoja for hoja in excel.sheet_names
    if hoja not in ["GraphData", "ÍNDICE DE CUADROS", "01-TOTAL DEL PAÍS"]
]

print("Cantidad de hojas de jurisdicciones:", len(hojas_jurisdicciones))
print(hojas_jurisdicciones)

Cantidad de hojas de jurisdicciones: 24
['02-CABA', '06-BUENOS AIRES', '10-CATAMARCA', '14-CÓRDOBA', '18-CORRIENTES', '22-CHACO', '26-CHUBUT', '30-ENTRE RÍOS', '34-FORMOSA', '38-JUJUY', '42-LA PAMPA', '46-LA RIOJA', '50-MENDOZA', '54-MISIONES', '58-NEUQUÉN', '62-RÍO NEGRO', '66-SALTA', '70-SAN JUAN', '74-SAN LUIS', '78-SANTA CRUZ', '82-SANTE FE', '86-SANTIAGO DEL ESTERO', '90-TUCUMÁN', '94-TIERRA DEL FUEGO']


In [20]:
# Leer la hoja de Buenos Aires sin encabezados automáticos
df_buenos_aires = pd.read_excel(
    POBLACION_FILE,
    sheet_name="06-BUENOS AIRES",
    engine="calamine",
    header=None
)

display(df_buenos_aires.head(12))

,0,1,2,3
0,Cuadro 1.3 Provincia de Buenos Aires. Població...,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,Año,Ambos sexos,Varones,Mujeres
4,2022,17409376,8582767,8826609
5,2023,17418577,8589613,8828964
6,2024,17431446,8599059,8832387
7,2025,17443372,8608571,8834801
8,2026,17453039,8616968,8836071
9,2027,17460685,8624351,8836334


In [21]:
# Lista para almacenar los resultados
poblacion_2023 = []

for hoja in hojas_jurisdicciones:

    # Leer cada cuadro sin encabezados automáticos
    df_hoja = pd.read_excel(
        POBLACION_FILE,
        sheet_name=hoja,
        engine="calamine",
        header=None
    )

    # Identificar la fila que contiene el año 2023
    fila_2023 = df_hoja[df_hoja.iloc[:, 0] == 2023]

    # Extraer población de ambos sexos
    poblacion = fila_2023.iloc[0, 1]

    # Guardar nombre de jurisdicción y población
    poblacion_2023.append({
        "jurisdiccion": hoja,
        "poblacion_2023": poblacion
    })

# Convertir resultados en DataFrame
df_poblacion = pd.DataFrame(poblacion_2023)

display(df_poblacion)

,jurisdiccion,poblacion_2023
0,02-CABA,3107894
1,06-BUENOS AIRES,17418577
2,10-CATAMARCA,436077
3,14-CÓRDOBA,3934514
4,18-CORRIENTES,1193970
5,22-CHACO,1194631
6,26-CHUBUT,598946
7,30-ENTRE RÍOS,1452721
8,34-FORMOSA,622428
9,38-JUJUY,796866


In [22]:
# Extraer los dos primeros dígitos del nombre de la hoja
df_poblacion["id_prov_indec_residencia"] = (
    df_poblacion["jurisdiccion"]
    .str.extract(r"^(\d{2})")
    .astype(int)
)

display(df_poblacion.head())

,jurisdiccion,poblacion_2023,id_prov_indec_residencia
0,02-CABA,3107894,2
1,06-BUENOS AIRES,17418577,6
2,10-CATAMARCA,436077,10
3,14-CÓRDOBA,3934514,14
4,18-CORRIENTES,1193970,18


In [23]:
df_poblacion["id_prov_norm"] = (
    df_poblacion["jurisdiccion"]
    .str.extract(r"^(\d{2})")[0]
)

In [24]:
print(gdf_provincias.columns.tolist())
display(gdf_provincias.head())

['id', 'nombre', 'nombre_completo', 'fuente', 'categoria', 'centroide', 'iso_id', 'iso_nombre', 'geometry']


,id,nombre,nombre_completo,fuente,categoria,centroide,iso_id,iso_nombre,geometry
0,02,Ciudad Autónoma de Buenos Aires,Ciudad Autónoma de Buenos Aires,IGN,Ciudad Autónoma,"{'lon': -58.445876325, 'lat': -34.614442065}",AR-C,Ciudad Autónoma de Buenos Aires,"POLYGON ((-58.45535 -34.52776, -58.45085 -34.5..."
1,58,Neuquén,Provincia del Neuquén,IGN,Provincia,"{'lon': -70.119897224, 'lat': -38.641982863}",AR-Q,Neuquén,"POLYGON ((-70.39341 -36.15624, -70.39499 -36.1..."
2,74,San Luis,Provincia de San Luis,IGN,Provincia,"{'lon': -66.025231271, 'lat': -33.761103538}",AR-D,San Luis,"POLYGON ((-67.05547 -31.85605, -67.00593 -31.8..."
3,82,Santa Fe,Provincia de Santa Fe,IGN,Provincia,"{'lon': -60.950687277, 'lat': -30.708822709}",AR-S,Santa Fe,"POLYGON ((-61.04639 -27.998, -58.884 -28, -58...."
4,46,La Rioja,Provincia de La Rioja,IGN,Provincia,"{'lon': -67.181757581, 'lat': -29.684937278}",AR-F,La Rioja,"POLYGON ((-68.52136 -27.81792, -68.51099 -27.8..."


### E. Normalizar los indices para unir las tablas a traves del identificador de provincia

In [25]:
# Códigos de dengue
casos_provincia["id_prov_norm"] = (
    casos_provincia["id_prov_indec_residencia"]
    .astype(str)
    .str.zfill(2)
)

# Códigos de población
df_poblacion["id_prov_norm"] = (
    df_poblacion["jurisdiccion"]
    .str.extract(r"^(\d{2})")[0]
)

# Códigos del GeoJSON
gdf_provincias["id_prov_norm"] = (
    gdf_provincias["id"].astype(str).str.zfill(2)
)

In [26]:
print("Códigos de dengue:", sorted(casos_provincia["id_prov_norm"].unique()))
print("Códigos de población:", sorted(df_poblacion["id_prov_norm"].unique()))
print("Códigos del mapa:", sorted(gdf_provincias["id_prov_norm"].unique()))

Códigos de dengue: ['02', '06', '10', '14', '18', '22', '26', '30', '34', '38', '42', '46', '50', '54', '58', '62', '66', '70', '74', '78', '82', '86', '90', '94']
Códigos de población: ['02', '06', '10', '14', '18', '22', '26', '30', '34', '38', '42', '46', '50', '54', '58', '62', '66', '70', '74', '78', '82', '86', '90', '94']
Códigos del mapa: ['02', '06', '10', '14', '18', '22', '26', '30', '34', '38', '42', '46', '50', '54', '58', '62', '66', '70', '74', '78', '82', '86', '90', '94']


In [27]:
df_analisis = casos_provincia.merge(
    df_poblacion[["id_prov_norm", "poblacion_2023"]],
    on="id_prov_norm",
    how="left",
    validate="one_to_one"
)

print("Provincias:", len(df_analisis))
print("Sin población:", df_analisis["poblacion_2023"].isna().sum())

display(df_analisis.head())

Provincias: 24
Sin población: 0


,id_prov_indec_residencia,provincia_residencia,casos,id_prov_norm,poblacion_2023
0,90,Tucumán,24172,90,1763239
1,82,Santa Fe,21328,82,3674361
2,22,Chaco,18369,22,1194631
3,86,Santiago del Estero,15843,86,1049610
4,66,Salta,15145,66,1473871


### F. Calculo de la incidencia

In [28]:
df_analisis["incidencia_100mil"] = (
    df_analisis["casos"] /
    df_analisis["poblacion_2023"]
) * 100000

df_analisis = df_analisis.sort_values(
    "incidencia_100mil",
    ascending=False
)

display(
    df_analisis[
        ["provincia_residencia", "casos",
         "poblacion_2023", "incidencia_100mil"]
    ].round(2)
)

,provincia_residencia,casos,poblacion_2023,incidencia_100mil
2,Chaco,18369,1194631,1537.63
3,Santiago del Estero,15843,1049610,1509.42
0,Tucumán,24172,1763239,1370.89
8,Formosa,6898,622428,1108.24
4,Salta,15145,1473871,1027.57
9,Jujuy,5410,796866,678.91
1,Santa Fe,21328,3674361,580.45
5,CABA,13292,3107894,427.69
10,Corrientes,3090,1193970,258.80
12,Catamarca,1073,436077,246.06


### Integración de datos epidemiológicos y poblacionales

Para analizar la distribución territorial del dengue, se integraron tres fuentes de información:

- **Casos de dengue:** registros del Ministerio de Salud de la Nación, correspondientes al evento Dengue y agrupados por provincia de residencia.
- **Población provincial:** estimaciones del INDEC al 1 de julio de 2023, considerando ambos sexos.
- **Geometrías provinciales:** archivo GeoJSON de provincias argentinas, utilizado para la representación cartográfica.

**Preparación e integración:**

1. Se conservaron los registros originales de dengue, sin eliminar automáticamente las filas con claves repetidas, debido a que no se pudo determinar si correspondían a duplicaciones o aportes independientes.
2. Se separaron los registros sin provincia identificada (código 99) y se excluyeron del análisis geográfico. Los registros con departamento desconocido, pero provincia identificada, se conservaron.
3. Se agregaron las cantidades de casos por provincia mediante suma.
4. Se extrajo la población estimada para 2023 de las 24 jurisdicciones, utilizando la columna «Ambos sexos» del archivo del INDEC.
5. Se normalizaron los códigos provinciales a dos dígitos y se verificó su correspondencia entre las tres fuentes.
6. Se integraron los casos y la población mediante una unión por código provincial, validando que existiera una correspondencia uno a uno.

Finalmente, se calculó la **incidencia por cada 100.000 habitantes**:

\[
\text{Incidencia} =
\frac{\text{Casos registrados}}{\text{Población estimada}}
\times 100.000
\]

**Consideraciones metodológicas:** los resultados deben interpretarse teniendo en cuenta que no se eliminaron registros potencialmente duplicados y que aún debe verificarse el alcance temporal exacto de los datos epidemiológicos antes de presentar la tasa como incidencia anual de 2023.

In [29]:
# Verificación temporal y de cantidades

print("Años presentes:")
print(sorted(df_dengue["anio_min"].unique()))

print("\nSemanas epidemiológicas:")
print(sorted(df_dengue["sepi_min"].unique()))

print("\nCantidad de semanas:")
print(df_dengue["sepi_min"].nunique())

print("\nCasos por año:")
display(
    df_dengue.groupby("anio_min")["cantidad"]
    .sum()
    .reset_index()
)

print("\nCasos por provincia identificada:")
print(df_nacional["cantidad"].sum())

print("\nCasos sin provincia identificada:")
print(df_desconocida["cantidad"].sum())

Años presentes:
[np.int64(2023)]

Semanas epidemiológicas:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52)]

Cantidad de semanas:
52

Casos por año:


,anio_min,cantidad
0,2023,146876



Casos por provincia identificada:
146408

Casos sin provincia identificada:
468


### G. Integración con datos de geometría provincial

In [30]:
gdf_analisis = gdf_provincias.merge(
    df_analisis,
    on="id_prov_norm",
    how="left",
    validate="one_to_one"
)

In [31]:
print("Cantidad de provincias:", len(gdf_analisis))

print(
    "Geometrías faltantes:",
    gdf_analisis.geometry.isna().sum()
)

print(
    "Incidencias faltantes:",
    gdf_analisis["incidencia_100mil"].isna().sum()
)

print("\nSistema de coordenadas:")
print(gdf_analisis.crs)

print("\nTipo de objeto:")
print(type(gdf_analisis))

Cantidad de provincias: 24
Geometrías faltantes: 0
Incidencias faltantes: 0

Sistema de coordenadas:
EPSG:4326

Tipo de objeto:
<class 'geopandas.geodataframe.GeoDataFrame'>


### H. Análisis preliminares de los datos

In [33]:
# Provincias con mayor cantidad absoluta de casos

display(
    df_analisis[
        ["provincia_residencia", "casos"]
    ]
    .sort_values("casos", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

,provincia_residencia,casos
0,Tucumán,24172
1,Santa Fe,21328
2,Chaco,18369
3,Santiago del Estero,15843
4,Salta,15145
5,CABA,13292
6,Buenos Aires,10443
7,Córdoba,8078
8,Formosa,6898
9,Jujuy,5410


In [34]:
# Comparar el ranking de casos absolutos con el de incidencia

df_comparacion = df_analisis.copy()

df_comparacion["puesto_casos"] = (
    df_comparacion["casos"]
    .rank(method="min", ascending=False)
    .astype(int)
)

df_comparacion["puesto_incidencia"] = (
    df_comparacion["incidencia_100mil"]
    .rank(method="min", ascending=False)
    .astype(int)
)

df_comparacion["cambio_posicion"] = (
    df_comparacion["puesto_casos"]
    - df_comparacion["puesto_incidencia"]
)

display(
    df_comparacion[
        [
            "provincia_residencia",
            "puesto_casos",
            "puesto_incidencia",
            "cambio_posicion"
        ]
    ]
    .sort_values("puesto_incidencia")
    .reset_index(drop=True)
)

,provincia_residencia,puesto_casos,puesto_incidencia,cambio_posicion
0,Chaco,3,1,2
1,Santiago del Estero,4,2,2
2,Tucumán,1,3,-2
3,Formosa,9,4,5
4,Salta,5,5,0
5,Jujuy,10,6,4
6,Santa Fe,2,7,-5
7,CABA,6,8,-2
8,Corrientes,11,9,2
9,Catamarca,13,10,3


## Análisis preliminares de los datos

### 1. Distribución de casos absolutos

El análisis de los casos registrados durante las semanas epidemiológicas 1 a 52 de 2023 muestra una distribución territorial heterogénea.

Las provincias con mayor cantidad de casos registrados fueron:

- Tucumán: 24.172 casos.
- Santa Fe: 21.328 casos.
- Chaco: 18.369 casos.
- Santiago del Estero: 15.843 casos.
- Salta: 15.145 casos.

Estos valores permiten identificar dónde se concentró la mayor cantidad absoluta de casos, pero no consideran las diferencias de población entre jurisdicciones.

### 2. Distribución de la incidencia por población

Para incorporar el tamaño poblacional, se calculó la incidencia acumulada registrada por cada 100.000 habitantes, utilizando las proyecciones de población del INDEC para 2023.

Las provincias con mayor incidencia fueron:

- Chaco: 1.537,63 casos cada 100.000 habitantes.
- Santiago del Estero: 1.509,42.
- Tucumán: 1.370,89.
- Formosa: 1.108,24.
- Salta: 1.027,57.

El ordenamiento por incidencia presenta diferencias respecto del ranking de casos absolutos.

### 3. Comparación preliminar de ambos indicadores

La comparación evidencia que la cantidad absoluta de casos y la incidencia relativa ofrecen perspectivas diferentes de la distribución territorial del dengue.

Algunos ejemplos relevantes:

- Chaco pasa del tercer puesto en cantidad de casos al primero en incidencia.
- Formosa pasa del noveno al cuarto puesto.
- Jujuy pasa del décimo al sexto puesto.
- Santa Fe desciende del segundo al séptimo puesto.
- Buenos Aires desciende del séptimo al decimocuarto puesto.

Estos resultados sugieren que considerar exclusivamente los casos absolutos puede ocultar diferencias importantes en la carga epidemiológica relativa entre provincias.

### 4. Limitaciones metodológicas

- El archivo utilizado contiene registros correspondientes a 2023 y las semanas epidemiológicas 1 a 52.
- Se identificaron 146.876 casos en el archivo, de los cuales 468 no tienen provincia identificada. Estos últimos se excluyeron del análisis provincial.
- El total del archivo no coincide con el total informado en el boletín epidemiológico oficial consultado. La causa de esta diferencia no pudo determinarse.
- Se conservaron los registros duplicados identificados durante la exploración, ya que no se encontró documentación suficiente para justificar su eliminación.
- La población utilizada corresponde a las proyecciones del INDEC para 2023. Por lo tanto, la incidencia es una estimación basada en esa población de referencia.
- Los resultados son descriptivos y no permiten establecer relaciones causales ni explicar los mecanismos de transmisión.

### 5. Pregunta que orienta el análisis espacial

¿La distribución territorial de los casos de dengue refleja también dónde fue mayor la carga epidemiológica relativa?

Esta pregunta orientará la siguiente etapa del proyecto, dedicada a identificar y comunicar patrones espaciales mediante mapas y visualizaciones comparativas.

In [35]:
# Guardar los datos preparados para el análisis espacial

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Tabla de análisis provincial
df_analisis.to_csv(
    PROCESSED_DIR / "dengue_analisis_provincial_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

# Datos geográficos integrados
gdf_analisis.to_file(
    PROCESSED_DIR / "dengue_provincias_2023.geojson",
    driver="GeoJSON"
)

print("Archivos procesados guardados correctamente.")

Archivos procesados guardados correctamente.
